In [7]:

from learning import *
from probabilistic_learning import *
from notebook import *
import sys
sys.path.append('./aima-python')

from logic import FolKB, expr, fol_fc_ask, fol_bc_ask,Expr
from probability import BayesNet, enumeration_ask

import numpy as np
import pandas as pd
from learning import NaiveBayesLearner, DataSet
from collections import Counter

In [1]:
def q3_naive_bayes_classification():
    """
    Question 1.3: Naive Bayes Classifiers
    Dataset 1: Iris
    Dataset 2: Spambase
    """
    
    import numpy as np
    import pandas as pd
    from collections import Counter
    
    print("="*60)
    print("NAIVE BAYES CLASSIFICATION")
    print("="*60)
    
    # ============================================================
    # DATASET 1: IRIS
    # ============================================================
    print("\n" + "="*60)
    print("[Dataset 1: Iris - Flower Classification]")
    print("="*60)
    
    # Load Iris directly from UCI
    iris_url = 'https://archive.ics.uci.edu/ml/machine-learning-databases/iris/iris.data'
    try:
        iris_df = pd.read_csv(iris_url, header=None, 
                              names=['sepal_length', 'sepal_width', 'petal_length', 'petal_width', 'class'])
    except:
        iris_df = pd.read_csv('iris.data', header=None, 
                              names=['sepal_length', 'sepal_width', 'petal_length', 'petal_width', 'class'])
    
    # Remove empty rows
    iris_df = iris_df.dropna()
    
    # Encode classes
    class_mapping = {label: idx for idx, label in enumerate(iris_df['class'].unique())}
    print(f"Class Mapping: {class_mapping}")
    
    X_iris = iris_df.iloc[:, :-1].values
    y_iris = iris_df['class'].map(class_mapping).values
    
    print(f"\nDataset Info:")
    print(f"  Total samples: {len(y_iris)}")
    print(f"  Features: {len(X_iris[0])}")
    print(f"  Feature names: {list(iris_df.columns[:-1])}")
    
    # Class distribution
    class_counts = Counter(y_iris)
    print(f"\nClass Distribution:")
    for cls, count in sorted(class_counts.items()):
        cls_name = [k for k, v in class_mapping.items() if v == cls][0]
        print(f"  {cls_name}: {count} samples ({count/len(y_iris)*100:.1f}%)")
    
    # ============================================================
    # 1.3.1: IRIS - Prior, Evidence, Likelihood
    # ============================================================
    print("\n" + "-"*60)
    print("PROBABILITY CALCULATIONS - IRIS")
    print("-"*60)
    
    # [1] Prior Probabilities
    print("\n[1] Prior Probabilities P(Class):")
    priors = {}
    for cls in sorted(class_counts.keys()):
        prior = class_counts[cls] / len(y_iris)
        priors[cls] = prior
        cls_name = [k for k, v in class_mapping.items() if v == cls][0]
        print(f"  P({cls_name}) = {class_counts[cls]}/{len(y_iris)} = {prior:.4f}")
    
    # [2] Evidence Probability
    print(f"\n[2] Evidence Probability (Petal Length):")
    petal_length = X_iris[:, 2]
    print(f"  Overall Mean: {np.mean(petal_length):.2f} cm")
    print(f"  Overall Std: {np.std(petal_length):.2f} cm")
    print(f"  Range: [{np.min(petal_length):.2f}, {np.max(petal_length):.2f}]")
    
    # [3] Likelihood
    print(f"\n[3] Likelihood P(Petal Length | Class):")
    likelihoods = {}
    for cls in sorted(class_counts.keys()):
        cls_mask = y_iris == cls
        cls_petal = X_iris[cls_mask, 2]
        mean_val = np.mean(cls_petal)
        std_val = np.std(cls_petal)
        likelihoods[cls] = (mean_val, std_val)
        cls_name = [k for k, v in class_mapping.items() if v == cls][0]
        print(f"  {cls_name}:")
        print(f"    Mean: {mean_val:.2f} cm, Std: {std_val:.2f} cm")
        print(f"    Range: [{np.min(cls_petal):.2f}, {np.max(cls_petal):.2f}]")
    
    # [4] Bayes' Theorem Example
    print(f"\n[4] Bayes' Theorem: P(Class | Petal Length = 4.0 cm)")
    print(f"  Formula: P(Class | Evidence) = P(Evidence | Class) × P(Class) / P(Evidence)")
    test_value = 4.0
    posteriors = {}
    for cls in sorted(class_counts.keys()):
        mean, std = likelihoods[cls]
        # Gaussian likelihood
        likelihood = (1 / (np.sqrt(2 * np.pi) * std)) * np.exp(-0.5 * ((test_value - mean) / std) ** 2)
        posterior_unnorm = likelihood * priors[cls]
        posteriors[cls] = posterior_unnorm
        cls_name = [k for k, v in class_mapping.items() if v == cls][0]
        print(f"  {cls_name}: {likelihood:.4f} × {priors[cls]:.4f} = {posterior_unnorm:.6f}")
    
    # Normalize posteriors
    total = sum(posteriors.values())
    print(f"\n  Normalized Posteriors (sum = 1.0):")
    for cls in sorted(posteriors.keys()):
        cls_name = [k for k, v in class_mapping.items() if v == cls][0]
        normalized = posteriors[cls] / total
        print(f"  P({cls_name} | PL=4.0) = {normalized:.4f}")
    
    predicted_class = max(posteriors, key=posteriors.get)
    pred_name = [k for k, v in class_mapping.items() if v == predicted_class][0]
    print(f"\n  Prediction: {pred_name} (highest posterior)")
    
    # ============================================================
    # DATASET 2: SPAMBASE
    # ============================================================
    print("\n\n" + "="*60)
    print("[Dataset 2: Spambase - Email Spam Detection]")
    print("="*60)
    
    # Load Spambase directly from UCI
    spam_url = 'https://archive.ics.uci.edu/ml/machine-learning-databases/spambase/spambase.data'
    try:
        spam_df = pd.read_csv(spam_url, header=None)
    except:
        spam_df = pd.read_csv('spambase.data', header=None)
    
    # Shuffle data
    spam_df = spam_df.sample(frac=1, random_state=42).reset_index(drop=True)
    
    X_spam = spam_df.iloc[:, :-1].values
    y_spam = spam_df.iloc[:, -1].values
    
    print(f"\nDataset Info:")
    print(f"  Total samples: {len(y_spam)}")
    print(f"  Features: {len(X_spam[0])}")
    print(f"  Classes: [0=not spam, 1=spam]")
    
    # Class distribution
    class_counts_s = Counter(y_spam)
    print(f"\nClass Distribution:")
    for cls, count in sorted(class_counts_s.items()):
        label = "not spam" if cls == 0 else "spam"
        print(f"  {label}: {count} samples ({count/len(y_spam)*100:.1f}%)")
    
    # ============================================================
    # 1.3.1: SPAMBASE - Prior, Evidence, Likelihood
    # ============================================================
    print("\n" + "-"*60)
    print("PROBABILITY CALCULATIONS - SPAMBASE")
    print("-"*60)
    
    # [1] Prior Probabilities
    print("\n[1] Prior Probabilities P(Class):")
    priors_s = {}
    for cls in sorted(class_counts_s.keys()):
        prior = class_counts_s[cls] / len(y_spam)
        priors_s[cls] = prior
        label = "not spam" if cls == 0 else "spam"
        print(f"  P({label}) = {class_counts_s[cls]}/{len(y_spam)} = {prior:.4f}")
    
    # [2] Evidence (Feature 0: word_freq_make)
    print(f"\n[2] Evidence Probability (Feature 0 - word_freq_make):")
    feature_0 = X_spam[:, 0]
    print(f"  Overall Mean: {np.mean(feature_0):.4f}")
    print(f"  Overall Std: {np.std(feature_0):.4f}")
    print(f"  Range: [{np.min(feature_0):.4f}, {np.max(feature_0):.4f}]")
    
    # [3] Likelihood
    print(f"\n[3] Likelihood P(word_freq_make | Class):")
    likelihoods_s = {}
    for cls in sorted(class_counts_s.keys()):
        cls_mask = y_spam == cls
        cls_feature = X_spam[cls_mask, 0]
        mean_val = np.mean(cls_feature)
        std_val = np.std(cls_feature)
        likelihoods_s[cls] = (mean_val, std_val)
        label = "not spam" if cls == 0 else "spam"
        print(f"  {label}:")
        print(f"    Mean: {mean_val:.4f}, Std: {std_val:.4f}")
    
    # [4] Bayes' Theorem Example
    print(f"\n[4] Bayes' Theorem: P(Class | word_freq_make = 0.5)")
    test_val = 0.5
    posteriors_s = {}
    for cls in sorted(class_counts_s.keys()):
        mean, std = likelihoods_s[cls]
        if std > 0:
            likelihood = (1 / (np.sqrt(2 * np.pi) * std)) * np.exp(-0.5 * ((test_val - mean) / std) ** 2)
        else:
            likelihood = 0
        posterior_unnorm = likelihood * priors_s[cls]
        posteriors_s[cls] = posterior_unnorm
        label = "not spam" if cls == 0 else "spam"
        print(f"  {label}: {likelihood:.6f} × {priors_s[cls]:.4f} = {posterior_unnorm:.6f}")
    
    # Normalize
    total_s = sum(posteriors_s.values())
    print(f"\n  Normalized Posteriors:")
    for cls in sorted(posteriors_s.keys()):
        label = "not spam" if cls == 0 else "spam"
        normalized = posteriors_s[cls] / total_s
        print(f"  P({label} | word_freq_make=0.5) = {normalized:.4f}")
    
    predicted_s = max(posteriors_s, key=posteriors_s.get)
    pred_label = "not spam" if predicted_s == 0 else "spam"
    print(f"\n  Prediction: {pred_label}")
    
    print("\n" + "="*60)
    print("✓ Dataset loading and probability calculations completed")
    print("="*60)



In [2]:

if __name__ == "__main__":
    q3_naive_bayes_classification()

NAIVE BAYES CLASSIFICATION

[Dataset 1: Iris - Flower Classification]
Class Mapping: {'Iris-setosa': 0, 'Iris-versicolor': 1, 'Iris-virginica': 2}

Dataset Info:
  Total samples: 150
  Features: 4
  Feature names: ['sepal_length', 'sepal_width', 'petal_length', 'petal_width']

Class Distribution:
  Iris-setosa: 50 samples (33.3%)
  Iris-versicolor: 50 samples (33.3%)
  Iris-virginica: 50 samples (33.3%)

------------------------------------------------------------
PROBABILITY CALCULATIONS - IRIS
------------------------------------------------------------

[1] Prior Probabilities P(Class):
  P(Iris-setosa) = 50/150 = 0.3333
  P(Iris-versicolor) = 50/150 = 0.3333
  P(Iris-virginica) = 50/150 = 0.3333

[2] Evidence Probability (Petal Length):
  Overall Mean: 3.76 cm
  Overall Std: 1.76 cm
  Range: [1.00, 6.90]

[3] Likelihood P(Petal Length | Class):
  Iris-setosa:
    Mean: 1.46 cm, Std: 0.17 cm
    Range: [1.00, 1.90]
  Iris-versicolor:
    Mean: 4.26 cm, Std: 0.47 cm
    Range: [3.00,

In [22]:
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

import sys
sys.stdout.flush()